In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers datasets peft accelerate


In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForMultipleChoice,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

possible_paths = [
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    "/kaggle/input/smart-mcq-solver-challenge/train.csv",
    "train.csv",
    "/content/train.csv"
]

train_path = next((path for path in possible_paths if os.path.exists(path)), None)

if train_path is None:
    raise FileNotFoundError("train.csv was not found. Add the competition dataset and run again.")

train = pd.read_csv(train_path)

label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
id2label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
choice_letters = ["A", "B", "C", "D", "E"]

train["label"] = train["answer"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Dataset shape:", train.shape)
train.head()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset shape: (2000, 9)


,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


# Q1. Label Encoding

Convert the `answer` column using:

- A = 0
- B = 1
- C = 2
- D = 3
- E = 4

Find the encoded label for row index 150.

In [4]:
q1_label = int(train.loc[150, "label"])

print("Answer letter:", train.loc[150, "answer"])
print("Encoded numeric label:", q1_label)


Answer letter: C
Encoded numeric label: 2


# Q2. Prompt–Option Formatting

For row index 0, create Option B using:

`str(prompt) + " [SEP] " + str(option_B)`

Find the exact character length.

In [5]:
row_0 = train.loc[0]

q2_input = str(row_0["prompt"]) + " [SEP] " + str(row_0["B"])
q2_length = len(q2_input)

print(q2_input)
print("Character length:", q2_length)


Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


# Q3. Single-Row MCQ Tokenization

Tokenize all five prompt-option inputs for row index 0 with:

- `padding="max_length"`
- `truncation=True`
- `max_length=128`
- `return_tensors="pt"`

Reshape the result to `[1, 5, 128]`.

In [6]:
row_0_inputs = [
    str(row_0["prompt"]) + " [SEP] " + str(row_0[letter])
    for letter in choice_letters
]

row_0_tokens = tokenizer(
    row_0_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

q3_input_ids = row_0_tokens["input_ids"].unsqueeze(0)
q3_attention_mask = row_0_tokens["attention_mask"].unsqueeze(0)

print("input_ids shape:", q3_input_ids.shape)
print("Second dimension:", q3_input_ids.shape[1])


input_ids shape: torch.Size([1, 5, 128])
Second dimension: 5


# Q4. Batch MCQ Tokenization

Tokenize the first 16 rows. The final tensor must have shape `[16, 5, 128]`.

Calculate the total number of token positions.

In [7]:
batch_input_ids = []
batch_attention_masks = []

for row_index in range(16):
    row = train.loc[row_index]

    formatted_choices = [
        str(row["prompt"]) + " [SEP] " + str(row[letter])
        for letter in choice_letters
    ]

    encoded = tokenizer(
        formatted_choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    batch_input_ids.append(encoded["input_ids"])
    batch_attention_masks.append(encoded["attention_mask"])

q4_input_ids = torch.stack(batch_input_ids)
q4_attention_mask = torch.stack(batch_attention_masks)

q4_total_positions = q4_input_ids.numel()

print("input_ids shape:", q4_input_ids.shape)
print("Total token positions:", q4_total_positions)

input_ids shape: torch.Size([16, 5, 128])
Total token positions: 10240


# Q5. Multiple-Choice Logits

Load `bert-base-uncased` with `AutoModelForMultipleChoice` and pass row index 0 through the model.

The output contains one logit for each answer choice.

In [8]:
set_seed(SEED)

base_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

base_model.eval()

with torch.no_grad():
    q5_outputs = base_model(
        input_ids=q3_input_ids,
        attention_mask=q3_attention_mask
    )

q5_logits = q5_outputs.logits

print("Logits:", q5_logits)
print("Logits shape:", q5_logits.shape)
print("Number of logits:", q5_logits.shape[1])


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits: tensor([[0.3439, 0.3109, 0.3058, 0.3230, 0.3445]])
Logits shape: torch.Size([1, 5])
Number of logits: 5


# Q6. Supervised Loss Tensor

Pass the correct encoded label for row index 0 along with the tokenized choices.

Inspect the number of dimensions in the returned loss tensor.


In [9]:
q6_label = torch.tensor([int(train.loc[0, "label"])], dtype=torch.long)

with torch.no_grad():
    q6_outputs = base_model(
        input_ids=q3_input_ids,
        attention_mask=q3_attention_mask,
        labels=q6_label
    )

q6_loss = q6_outputs.loss

print("Loss:", q6_loss)
print("Loss tensor dimensions:", q6_loss.dim())


Loss: tensor(1.6243)
Loss tensor dimensions: 0


# Q7. LoRA Trainable Parameters

Apply LoRA using:

- `r=8`
- `lora_alpha=16`
- `target_modules=["query", "value"]`
- `lora_dropout=0.1`
- `bias="none"`
- `task_type=TaskType.SEQ_CLS`

Count all parameters for which `requires_grad=True`.


In [18]:
!pip uninstall -y torchao peft
!pip install -q --no-deps peft==0.14.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 2.4 MB/s eta 0:00:0000:0100:01


In [19]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

q7_base_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

q7_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

q7_lora_model = get_peft_model(
    q7_base_model,
    q7_lora_config
)

q7_trainable_parameters = sum(
    p.numel()
    for p in q7_lora_model.parameters()
    if p.requires_grad
)

print(q7_trainable_parameters)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


295681


# Q8. Hugging Face Dataset Preparation

Create a Hugging Face `Dataset` from the first 100 rows.

Each item must contain:

- `input_ids` with shape `[5, 128]`
- `attention_mask` with shape `[5, 128]`
- `labels` containing the encoded answer


In [11]:
first_100_df = train.iloc[:100].copy()
q8_dataset = Dataset.from_pandas(first_100_df, preserve_index=False)

def tokenize_mcq_128(example):
    formatted_choices = [
        str(example["prompt"]) + " [SEP] " + str(example[letter])
        for letter in choice_letters
    ]

    encoded = tokenizer(
        formatted_choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

q8_tokenized_dataset = q8_dataset.map(
    tokenize_mcq_128,
    remove_columns=q8_dataset.column_names
)

q8_first_item = q8_tokenized_dataset[0]

print("input_ids shape:", np.array(q8_first_item["input_ids"]).shape)
print("Number of tokenized choices:", len(q8_first_item["input_ids"]))
print("Label:", q8_first_item["labels"])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape: (5, 128)
Number of tokenized choices: 5
Label: 1


# Q9. Tiny LoRA Fine-Tuning

Fine-tune on the first 32 rows with:

- `max_length=64`
- `per_device_train_batch_size=4`
- `gradient_accumulation_steps=1`
- `max_steps=4`

In [20]:
set_seed(SEED)

first_32_df = train.iloc[:32].copy()
q9_dataset = Dataset.from_pandas(first_32_df, preserve_index=False)

def tokenize_mcq_64(example):
    formatted_choices = [
        str(example["prompt"]) + " [SEP] " + str(example[letter])
        for letter in choice_letters
    ]

    encoded = tokenizer(
        formatted_choices,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

q9_tokenized_dataset = q9_dataset.map(
    tokenize_mcq_64,
    remove_columns=q9_dataset.column_names
)

class MultipleChoiceDataCollator:
    def __call__(self, features):
        return {
            "input_ids": torch.tensor(
                [feature["input_ids"] for feature in features],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                [feature["attention_mask"] for feature in features],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                [feature["labels"] for feature in features],
                dtype=torch.long
            )
        }

set_seed(SEED)

q9_base_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

q9_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

q9_model = get_peft_model(
    q9_base_model,
    q9_lora_config
)

training_arguments = TrainingArguments(
    output_dir="./milestone4_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED
)

trainer = Trainer(
    model=q9_model,
    args=training_arguments,
    train_dataset=q9_tokenized_dataset,
    data_collator=MultipleChoiceDataCollator()
)

q9_train_result = trainer.train()

print("Final global_step:", trainer.state.global_step)


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.606301
2,1.597858
3,1.659202
4,1.640764


Final global_step: 4


# Q10. Probability Assigned to Option E

Use the fine-tuned model from Q9 for row index 0.

Apply softmax to the five logits and report the probability assigned to Option E, rounded to four decimal places.

In [21]:
row_0 = train.loc[0]

q10_inputs = [
    str(row_0["prompt"]) + " [SEP] " + str(row_0[letter])
    for letter in choice_letters
]

q10_encoded = tokenizer(
    q10_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

q10_device = trainer.model.device

q10_input_ids = q10_encoded["input_ids"].unsqueeze(0).to(q10_device)
q10_attention_mask = q10_encoded["attention_mask"].unsqueeze(0).to(q10_device)

trainer.model.eval()

with torch.no_grad():
    q10_outputs = trainer.model(
        input_ids=q10_input_ids,
        attention_mask=q10_attention_mask
    )

q10_logits = q10_outputs.logits
q10_probabilities = torch.softmax(q10_logits, dim=-1)
q10_option_e_probability = q10_probabilities[0, 4].item()

print("Logits:", q10_logits)
print("Probabilities:", q10_probabilities)
print("Option E probability:", round(q10_option_e_probability, 4))


Logits: tensor([[0.3591, 0.3484, 0.3564, 0.3212, 0.3214]])
Probabilities: tensor([[0.2036, 0.2014, 0.2030, 0.1960, 0.1960]])
Option E probability: 0.196
